In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge
from tqdm import tqdm
import geopandas as gpd

In [2]:
# Setting up Working Directory
WORK_DIR = Path("/Users/muhammadabdul/Desktop/penn/earth_genome_internship/climate_trace")

# Folder containing input feature TIFFs 
input_tiff_folder = WORK_DIR / "data/KS_barton_lot_pond/county_polyAlphaEarth_sp"

# Path to trained Random Forest pickle
model = '20260309_150216'
rf_model_path = WORK_DIR / f"sub_facility_discrimination_KS_sp/model/{model}/model.pkl"

# Output folder for prediction mask TIFFs
output_mask_folder = WORK_DIR / f"sub_facility_discrimination_KS_sp/inference/{model}"
output_mask_folder.mkdir(parents=True, exist_ok=True)

In [3]:
# load model and find tifffs
with open(rf_model_path, "rb") as f:
    rf_model = pickle.load(f)
input_tiffs = sorted(input_tiff_folder.glob("*.tif"))

In [4]:
# Define mapping from string labels to integer codes for saving as TIFF
class_to_code = {'lot': 2,'pond': 1, 'other': 0}

for tiff_path in tqdm(input_tiffs):

    # Read TIFF and convert to DataFrame for model input
    with rasterio.open(tiff_path) as src:
        x = src.read().astype(np.float32)
    
    # Reshape x to (num_pixels, num_bands) and create DataFrame with appropriate column names
    x_df = pd.DataFrame(x.reshape(x.shape[0], -1).T, columns=[f'band_{i + 1}' for i in range(x.shape[0])])

    # Predict with the model
    predictions = rf_model.predict(x_df)

    # Reshape predictions back to original spatial dimensions
    pred_mask = predictions.reshape(x.shape[1], x.shape[2])
    
    # Map string labels to integer codes for saving as TIFF
    pred_mask = np.where(pred_mask == "other", class_to_code['other'], pred_mask)
    pred_mask = np.where(pred_mask == "lot", class_to_code['lot'], pred_mask)
    pred_mask = np.where(pred_mask == "pond", class_to_code['pond'], pred_mask)

    # saving probabilities in pred_mask as sperate band in the output TIFF
    pred_probs = rf_model.predict_proba(x_df)
    pred_probs = np.max(pred_probs, axis=1).reshape(x.shape[1], x.shape[2])

    # Save prediction mask as a new TIFF
    output_path = output_mask_folder / f"{tiff_path.stem}.tif"
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=pred_mask.shape[0],
        width=pred_mask.shape[1],
        count=2,
        dtype=np.float32,
        crs=src.crs,
        transform=src.transform,
    ) as dst:
        dst.write(pred_mask, 1)
        dst.write(pred_probs, 2)

100%|██████████| 21/21 [00:07<00:00,  2.78it/s]


In [5]:
# find all prediction mask TIFFs
mask_files = sorted(output_mask_folder.glob("*.tif"))
datasets = [rasterio.open(path) for path in mask_files]

# Use rasterio's merge function to combine all prediction masks into one
merged_arr, merged_transform = merge(datasets, dtype="float32")
merged_profile = datasets[0].profile.copy()

# Update profile for merged TIFF
merged_profile.update(
    driver="GTiff",
    height=merged_arr.shape[1],
    width=merged_arr.shape[2],
    transform=merged_transform,
    count=2,
    dtype="float32",
    nodata=255,
    compress="lzw",
)

# Save merged prediction mask and probabilities as a new TIFF
merged_path = output_mask_folder / "merged.tif"
with rasterio.open(merged_path, "w", **merged_profile) as dst:
    dst.write(merged_arr[0].astype("float32"), 1)
    dst.write(merged_arr[1].astype("float32"), 2)
    dst.set_band_description(1, "prediction")
    dst.set_band_description(2, "probability")

In [ ]:
# convert predictions that do not lie within the spatial prior polygons to "other" class (code 0)
sp_path = WORK_DIR / 'data' / 'KS_barton_lot_pond' / 'spatial_prior.geojson'
sp_polys = gpd.read_file(sp_path)
sp_polys["geometry"] = sp_polys.geometry.envelope
sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')

with rasterio.open(merged_path) as src:
    pred_mask = src.read(1)
    pred_probs = src.read(2)
    transform = src.transform
    crs = src.crs

    # Create a grid of pixel coordinates
    height, width = pred_mask.shape
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rasterio.transform.xy(transform, rows, cols)
    points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(xs.flatten(), ys.flatten()), crs=crs)

    # Check which points fall within the spatial prior polygons
    within_prior = points.within(sp_polys.union_all())

    # Update predictions to "other" (code 0) for points outside the spatial prior
    pred_mask_flat = pred_mask.flatten()
    pred_mask_flat[~within_prior] = class_to_code['other']
    pred_mask = pred_mask_flat.reshape(pred_mask.shape)

    # convert probabilities to nan for points outside the spatial prior
    pred_probs_flat = pred_probs.flatten()
    pred_probs_flat[~within_prior] = np.nan
    pred_probs = pred_probs_flat.reshape(pred_probs.shape)

    # Save the updated prediction mask as a new TIFF
    updated_path = output_mask_folder / "merged_processed.tif"
    with rasterio.open(updated_path, "w", driver="GTiff", height=pred_mask.shape[0], width=pred_mask.shape[1], count=2, dtype="float32", crs=crs, transform=transform) as dst:
        dst.write(pred_mask.astype("float32"), 1)
        dst.write(pred_probs.astype("float32"), 2)
        dst.set_band_description(1, "updated_prediction")
        dst.set_band_description(2, "probability")

/var/folders/zf/m0p4r__d7gb0cphf85xj8z180000gn/T/ipykernel_63857/242144432.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sp_polys = gpd.GeoDataFrame(geometry=sp_polys.buffer(0.001), crs='epsg:4326')
